# 🧠 EX49: การวิเคราะห์ผลลัพธ์ของ YOLO และการวินิจฉัยการฝึกฝน (YOLO Result Analysis & Training Diagnostics)

ยินดีต้อนรับนักเรียนทุกคน! วันนี้เราจะมาเรียนรู้กระบวนการสำคัญในการวิเคราะห์บันทึกการฝึกฝน (training logs) เพื่อวินิจฉัยพฤติกรรมการเรียนรู้ของแบบจำลอง YOLO ในการฝึกฝนตัวตรวจจับวัตถุแบบกำหนดเอง (custom object detectors) คุณไม่สามารถปล่อยให้แบบจำลองทำงานไปแบบสุ่มสี่สุ่มห้าได้ แต่คุณต้องตรวจสอบการเปลี่ยนแปลงระหว่างการฝึกฝนเพื่อตัดสินใจเกี่ยวกับไฮเปอร์พารามิเตอร์ (hyperparameters), การเพิ่มข้อจำกัดความเสถียร (regularization) และขนาดความสามารถของแบบจำลอง (model capacity) ได้อย่างถูกต้อง

## 1. เจาะลึกกระบวนการ: ฟังก์ชันการสูญเสีย (Loss Functions) ใน YOLO แบบ Anchor-Free

รุ่นของ YOLO ยุคใหม่ (เช่น YOLOv8 และ YOLO11) เป็นตัวตรวจจับแบบ **Anchor-Free** (ไม่มีกล่องสมอ) มาเปรียบเทียบความแตกต่างกับสถาปัตยกรรมแบบ anchor-based รุ่นเก่ากัน:
- **แบบ Anchor-Based (เช่น YOLOv3, YOLOv4, YOLOv5):** แบบจำลองเหล่านี้ขึ้นอยู่กับรูปทรงของกล่องขอบเขตที่กำหนดไว้ล่วงหน้า (สมอหรือ anchors) เครือข่ายจะทำนายระยะห่างที่เลี้ยวเบนไปจากสมอเหล่านี้ หากเลือกสมอไม่เหมาะสมกับขนาดวัตถุจริง อาจทำให้ประสิทธิภาพการถดถอยกล่องแย่ลงอย่างมาก และจำเป็นต้องทำการจัดกลุ่มสมอ (anchor clustering) ด้วยมือก่อนใช้งาน
- **แบบ Anchor-Free (เช่น YOLOv8, YOLO11):** แบบจำลองเหล่านี้ทำนายขอบเขตของกล่องขอบเขตโดยตรงจากจุดพิกัดกริด (ระยะห่างจากจุดไปยังขอบซ้าย, บน, ขวา, ล่าง) ทำให้มีความยืดหยุ่นและมีความเสถียรมากกว่ามาก

ในระหว่างการฝึกฝน YOLO จะติดตามองค์ประกอบการสูญเสียหลักสามตัวในไฟล์ `results.csv`:
1. **การสูญเสียพิกัดกล่อง (Box Loss - `train/box_loss`, `val/box_loss`):** คำนวณโดยใช้ Complete IoU (CIoU) loss ซึ่งทำหน้าที่วัดว่าพิกัดของกล่องขอบเขตที่ทำนายตรงกับพิกัดจริงมากน้อยเพียงใด
2. **การสูญเสียการจำแนกประเภท (Classification Loss - `train/cls_loss`, `val/cls_loss`):** คำนวณโดยใช้ Binary Cross-Entropy (BCE) loss ซึ่งทำหน้าที่วัดความถูกต้องของการจำแนกคลาสในแต่ละเซลล์กริด
3. **การสูญเสียโฟคัลการแจกแจงแบบดีเอฟแอล (Distribution Focal Loss - `train/dfl_loss`, `val/dfl_loss`):** ในแบบจำลองแบบ anchor-free พิกัดกล่องขอบเขตจะถูกจำลองด้วยการแจกแจงความน่าจะเป็นแบบต่อเนื่อง DFL จะช่วยให้แบบจำลองเรียนรู้ตำแหน่งขอบเขตเทียบกับจุดศูนย์กลางเป้าหมายได้อย่างแม่นยำยิ่งขึ้น โดยเฉพาะอย่างยิ่งภายใต้สภาพที่วัตถุบังกันหรือขอบเขตพร่ามัว

การสูญเสียรวม (Total Loss) คือผลรวมถ่วงน้ำหนักขององค์ประกอบทั้งสามนี้ เราจะเฝ้าติดตามทั้งการสูญเสียจากการฝึกฝน (training loss) และการสูญเสียการตรวจสอบความถูกต้อง (validation loss) เพื่อประเมินการลู่เข้าหาคำตอบ (convergence)

## 2. การอธิบายตัวชี้วัดประสิทธิภาพตามหลักการพื้นฐาน

ในการตรวจสอบความถูกต้องของแบบจำลอง YOLO เราจะใช้ตัวชี้วัดที่ได้จากพื้นที่ทับซ้อนระหว่างกล่องทำนายและกล่องจริง:

### A. ความแม่นยำและการระลึก (Precision & Recall)
- **ความแม่นยำ (Precision - P):** จากกล่องทำนายที่แบบจำลองสร้างขึ้นทั้งหมด มีกล่องที่ถูกต้องกี่เปอร์เซ็นต์?
  $$\text{Precision} = \frac{\text{True Positives (TP)}}{\text{True Positives (TP)} + \text{False Positives (FP)}}$$
  *ความแม่นยำสูงแสดงว่าเกิดผลบวกลวงน้อย (อัตราสัญญาณเตือนผิดต่ำ)* 

- **การระลึก (Recall - R):** จากวัตถุจริงทั้งหมดที่มีอยู่ในชุดข้อมูล แบบจำลองสามารถค้นพบกี่เปอร์เซ็นต์?
  $$\text{Recall} = \frac{\text{True Positives (TP)}}{\text{True Positives (TP)} + \text{False Negatives (FN)}}$$
  *การระลึกสูงแสดงว่าเกิดผลลบลวงน้อย (พลาดการตรวจจับวัตถุจริงน้อยมาก)* 

### B. ค่าความแม่นยำเฉลี่ยเฉลี่ย (Mean Average Precision - mAP)
ค่าความแม่นยำเฉลี่ย (Average Precision - AP) คือพื้นที่ใต้เส้นโค้ง Precision-Recall และค่าความแม่นยำเฉลี่ยเฉลี่ย (mAP) คือค่าเฉลี่ยของ AP ของทุกคลาส
- **mAP@0.5 (หรือ mAP50):** ค่า mAP ที่คำนวณที่เกณฑ์ค่าจุดตัดส่วนด้วยจุดรวม (IoU) คงที่เท่ากับ $0.5$ หากกล่องทำนายทับซ้อนกับกล่องจริงตั้งแต่ $\ge 50\%$, จะถือว่าการทำนายนั้นถูกต้อง นี่เป็นตัวชี้วัดที่ผ่อนปรนซึ่งใช้ตรวจสอบว่าแบบจำลองระบุพิกัดโดยทั่วไปของวัตถุได้ถูกต้องหรือไม่
- **mAP@0.5:0.95 (หรือ mAP50-95):** ค่า mAP ที่เฉลี่ยจากเกณฑ์ IoU 10 ระดับที่แตกต่างกัน: $0.5, 0.55, 0.6, \dots, 0.95$ นี่คือตัวชี้วัดหลักที่ใช้ในการแข่งขันของ COCO มีความเข้มงวดสูงกว่ามากและให้คะแนนสูงแก่กล่องทำนายที่มีพิกัดขอบเขตตรงกับพิกัดจริงอย่างแม่นยำมาก

In [ ]:
# Back up or checkpoint this section of code before starting to modify the large file.
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

def generate_mock_data(output_path: str, mode: str = 'normal', num_epochs: int = 50):
    """
    Generates a mock results.csv file simulating YOLO training outputs.
    """
    epochs = np.arange(1, num_epochs + 1)
    time_per_epoch = 5.0 + np.random.normal(0, 0.2, num_epochs)
    
    train_box = 1.5 * np.exp(-epochs / 15) + 0.1 + np.random.normal(0, 0.01, num_epochs)
    train_cls = 2.0 * np.exp(-epochs / 12) + 0.05 + np.random.normal(0, 0.01, num_epochs)
    train_dfl = 1.0 * np.exp(-epochs / 18) + 0.02 + np.random.normal(0, 0.005, num_epochs)
    
    if mode == 'normal':
        val_box = 1.5 * np.exp(-epochs / 14) + 0.15 + np.random.normal(0, 0.02, num_epochs)
        val_cls = 2.0 * np.exp(-epochs / 11) + 0.1 + np.random.normal(0, 0.02, num_epochs)
        val_dfl = 1.0 * np.exp(-epochs / 16) + 0.04 + np.random.normal(0, 0.01, num_epochs)
        mAP50 = 0.9 * (1.0 - np.exp(-epochs / 8)) + np.random.normal(0, 0.005, num_epochs)
        mAP50_95 = 0.7 * (1.0 - np.exp(-epochs / 9)) + np.random.normal(0, 0.005, num_epochs)
    else:
        val_box = []
        val_cls = []
        val_dfl = []
        mAP50 = []
        mAP50_95 = []
        for e in epochs:
            base_box = 1.5 * np.exp(-e / 14) + 0.15
            base_cls = 2.0 * np.exp(-e / 11) + 0.1
            base_dfl = 1.0 * np.exp(-e / 16) + 0.04
            base_mAP50 = 0.9 * (1.0 - np.exp(-e / 8))
            base_mAP50_95 = 0.7 * (1.0 - np.exp(-e / 9))
            if e > 25:
                drift = 0.03 * (e - 25)
                base_box += drift
                base_cls += drift * 1.5
                base_dfl += drift * 0.5
                base_mAP50 -= 0.005 * (e - 25)
                base_mAP50_95 -= 0.008 * (e - 25)
            val_box.append(base_box + np.random.normal(0, 0.01))
            val_cls.append(base_cls + np.random.normal(0, 0.01))
            val_dfl.append(base_dfl + np.random.normal(0, 0.005))
            mAP50.append(max(0.0, min(1.0, base_mAP50 + np.random.normal(0, 0.005))))
            mAP50_95.append(max(0.0, min(1.0, base_mAP50_95 + np.random.normal(0, 0.005))))
        val_box = np.array(val_box)
        val_cls = np.array(val_cls)
        val_dfl = np.array(val_dfl)
        mAP50 = np.array(mAP50)
        mAP50_95 = np.array(mAP50_95)
        
    precision = 0.85 * mAP50 + np.random.normal(0, 0.01, num_epochs)
    recall = 0.8 * mAP50 + np.random.normal(0, 0.01, num_epochs)
    lr_pg0 = 0.01 * (1.0 - epochs / num_epochs)
    
    df = pd.DataFrame({
        'epoch': epochs,
        'time': time_per_epoch,
        'train/box_loss': train_box,
        'train/cls_loss': train_cls,
        'train/dfl_loss': train_dfl,
        'metrics/precision(B)': precision,
        'metrics/recall(B)': recall,
        'metrics/mAP50(B)': mAP50,
        'metrics/mAP50-95(B)': mAP50_95,
        'val/box_loss': val_box,
        'val/cls_loss': val_cls,
        'val/dfl_loss': val_dfl,
        'lr/pg0': lr_pg0,
        'lr/pg1': lr_pg0,
        'lr/pg2': lr_pg0
    })
    df.to_csv(output_path, index=False)

# Create directories if needed
os.makedirs('runs/detect/train_mock_normal', exist_ok=True)
os.makedirs('runs/detect/train_mock_overfit', exist_ok=True)

normal_path = 'runs/detect/train_mock_normal/results.csv'
overfit_path = 'runs/detect/train_mock_overfit/results.csv'

generate_mock_data(normal_path, mode='normal', num_epochs=45)
generate_mock_data(overfit_path, mode='overfit', num_epochs=45)
print("Mock data generated successfully.")

In [ ]:
# Analysis functions from solution.py
def check_diagnostics(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    
    # Calculate total training and validation losses
    df['total_train_loss'] = df[['train/box_loss', 'train/cls_loss', 'train/dfl_loss']].sum(axis=1)
    df['total_val_loss'] = df[['val/box_loss', 'val/cls_loss', 'val/dfl_loss']].sum(axis=1)
    
    # Best epoch detection
    best_row_idx = df['metrics/mAP50-95(B)'].idxmax()
    best_epoch = df.loc[best_row_idx, 'epoch']
    best_map = df.loc[best_row_idx, 'metrics/mAP50-95(B)']
    
    # Overfitting check (last 5 epochs)
    recent = df.tail(5)
    epochs = recent['epoch'].values
    train_slope, _ = np.polyfit(epochs, recent['total_train_loss'].values, 1)
    val_slope, _ = np.polyfit(epochs, recent['total_val_loss'].values, 1)
    
    print(f"--- Analysis for {os.path.basename(os.path.dirname(csv_path))} ---")
    print(f"Best Epoch (by mAP50-95): {int(best_epoch)} (Value: {best_map:.4f})")
    print(f"Train loss slope (last 5 epochs): {train_slope:.5f}")
    print(f"Val loss slope (last 5 epochs): {val_slope:.5f}")
    
    if train_slope < -1e-5 and val_slope > 1e-5:
        print("WARNING: Overfitting detected! Train loss is falling, but val loss is rising!")
    else:
        print("No severe overfitting detected in the final epochs.")
    print()

check_diagnostics(normal_path)
check_diagnostics(overfit_path)

In [ ]:
# Plotting the training curves
def plot_training_curves(normal_csv, overfit_csv):
    df_normal = pd.read_csv(normal_csv)
    df_normal.columns = df_normal.columns.str.strip()
    df_normal['total_train_loss'] = df_normal[['train/box_loss', 'train/cls_loss', 'train/dfl_loss']].sum(axis=1)
    df_normal['total_val_loss'] = df_normal[['val/box_loss', 'val/cls_loss', 'val/dfl_loss']].sum(axis=1)
    
    df_overfit = pd.read_csv(overfit_csv)
    df_overfit.columns = df_overfit.columns.str.strip()
    df_overfit['total_train_loss'] = df_overfit[['train/box_loss', 'train/cls_loss', 'train/dfl_loss']].sum(axis=1)
    df_overfit['total_val_loss'] = df_overfit[['val/box_loss', 'val/cls_loss', 'val/dfl_loss']].sum(axis=1)
    
    fig, axs = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Normal Run Losses
    axs[0, 0].plot(df_normal['epoch'], df_normal['total_train_loss'], label='Train Loss', color='#1f77b4', linewidth=2)
    axs[0, 0].plot(df_normal['epoch'], df_normal['total_val_loss'], label='Val Loss', color='#ff7f0e', linewidth=2)
    axs[0, 0].set_title('Normal Run - Loss Curves', fontsize=14, fontweight='bold')
    axs[0, 0].set_xlabel('Epochs', fontsize=12)
    axs[0, 0].set_ylabel('Total Loss', fontsize=12)
    axs[0, 0].legend(fontsize=11)
    axs[0, 0].grid(True, linestyle='--', alpha=0.6)
    
    # 2. Overfit Run Losses
    axs[0, 1].plot(df_overfit['epoch'], df_overfit['total_train_loss'], label='Train Loss', color='#1f77b4', linewidth=2)
    axs[0, 1].plot(df_overfit['epoch'], df_overfit['total_val_loss'], label='Val Loss (Overfitting)', color='#d62728', linewidth=2)
    axs[0, 1].axvline(x=25, color='#7f7f7f', linestyle=':', label='Overfitting Begins')
    axs[0, 1].set_title('Overfitting Run - Loss Curves', fontsize=14, fontweight='bold')
    axs[0, 1].set_xlabel('Epochs', fontsize=12)
    axs[0, 1].set_ylabel('Total Loss', fontsize=12)
    axs[0, 1].legend(fontsize=11)
    axs[0, 1].grid(True, linestyle='--', alpha=0.6)
    
    # 3. Normal Run mAP
    axs[1, 0].plot(df_normal['epoch'], df_normal['metrics/mAP50(B)'], label='mAP@0.5', color='#2ca02c', linewidth=2)
    axs[1, 0].plot(df_normal['epoch'], df_normal['metrics/mAP50-95(B)'], label='mAP@0.5:0.95', color='#9467bd', linewidth=2)
    axs[1, 0].set_title('Normal Run - Validation mAP', fontsize=14, fontweight='bold')
    axs[1, 0].set_xlabel('Epochs', fontsize=12)
    axs[1, 0].set_ylabel('mAP Score', fontsize=12)
    axs[1, 0].legend(fontsize=11)
    axs[1, 0].grid(True, linestyle='--', alpha=0.6)
    
    # 4. Overfit Run mAP
    axs[1, 1].plot(df_overfit['epoch'], df_overfit['metrics/mAP50(B)'], label='mAP@0.5', color='#2ca02c', linewidth=2)
    axs[1, 1].plot(df_overfit['epoch'], df_overfit['metrics/mAP50-95(B)'], label='mAP@0.5:0.95 (Degrading)', color='#9467bd', linewidth=2)
    axs[1, 1].axvline(x=25, color='#7f7f7f', linestyle=':', label='Overfitting Begins')
    axs[1, 1].set_title('Overfitting Run - Validation mAP', fontsize=14, fontweight='bold')
    axs[1, 1].set_xlabel('Epochs', fontsize=12)
    axs[1, 1].set_ylabel('mAP Score', fontsize=12)
    axs[1, 1].legend(fontsize=11)
    axs[1, 1].grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    # Save the output chart to the exercise directory
    plt.savefig('loss_curve_check.png', dpi=300)
    plt.show()

plot_training_curves(normal_path, overfit_path)

## 3. การวินิจฉัยเส้นโค้งการฝึกฝน (Diagnosing Training Curves)

มาสรุปลักษณะโปรไฟล์ของเส้นโค้งการฝึกฝนมาตรฐานจากมุมมองของวิศวกรรมคอมพิวเตอร์วิทัศน์กัน:

### 1. การเรียนรู้เกินแผนภูมิ / การฟิตเกิน (Overfitting - สถานการณ์ B)
- **สิ่งที่คุณเห็น:** การสูญเสียของการฝึกฝน (training loss) ยังคงลดลงอย่างราบรื่น แต่เส้นโค้งการสูญเสียของการตรวจสอบ (validation loss) จะถึงจุดต่ำสุดแล้วเริ่มสูงขึ้น ในเวลาเดียวกัน ตัวชี้วัดของการตรวจสอบอย่าง `mAP@0.5:0.95` จะถึงจุดสูงสุดแล้วเริ่มลดลง
- **ทำไมจึงเกิดขึ้น:** แบบจำลองกำลังจดจำคุณลักษณะเฉพาะของความถี่สูง (หรือสัญญาณรบกวน) จากชุดข้อมูลฝึกฝนแทนที่จะเรียนรู้รูปแบบทั่วไป พฤติกรรมนี้มักเกิดขึ้นเมื่อความสามารถของแบบจำลอง (model capacity) สูงเกินไปสำหรับขนาดของชุดข้อมูล หรือเมื่อการฝึกฝนรันเป็นจำนวนรอบ (epochs) มากเกินไปโดยไม่มีการเพิ่มข้อจำกัดความเสถียร (regularization) ที่เพียงพอ
- **แนวทางการแก้ไขของศาสตราจารย์:**
  - เพิ่มการสลายน้ำหนัก (`weight_decay=0.0005`)
  - ปรับปรุงการเพิ่มขยายข้อมูล (data augmentations): เพิ่มอัตราของ `mosaic`, เพิ่ม `mixup`, หรือใช้ประโยชน์จาก `degrees` (การหมุนรูปภาพ)
  - ลดขนาดแบบจำลองลง (เช่น ลดจาก `yolo26s` ลงมาเป็น `yolo26n`)
  - เปิดใช้งานการหยุดก่อนกำหนด (`patience=50` หรือใกล้เคียง)

### 2. การเรียนรู้ขาดแผนภูมิ / การฟิตขาด (Underfitting)
- **สิ่งที่คุณเห็น:** ทั้งการสูญเสียของการฝึกฝนและการตรวจสอบยังคงอยู่ในระดับสูง หรือลู่เข้าหาค่าที่สูงจนยอมรับไม่ได้ ตัวชี้วัด validation mAP แบนราบในระดับต่ำ
- **ทำไมจึงเกิดขึ้น:** แบบจำลองขาดความสามารถในการแสดงความซับซ้อนของชุดข้อมูลเป้าหมาย หรือตัวปรับค่าพารามิเตอร์ (optimizer) ติดอยู่ในจุดต่ำสุดท้องถิ่นที่แย่ (poor local minimum)
- **แนวทางการแก้ไขของศาสตราจารย์:**
  - ใช้แบบจำลองที่มีขนาดใหญ่ขึ้น (เช่น เพิ่มขนาดจาก `yolo26n` เป็น `yolo26s` หรือ `yolo26m`)
  - ลดความเข้มงวดในการป้องกันการฟิตเกิน (เช่น ลดการสลายน้ำหนัก หรือลดอัตราการขยายข้อมูล)
  - เพิ่มจำนวนรอบการฝึกฝน (`epochs`)
  - ตรวจสอบพารามิเตอร์อัตราการเรียนรู้ (เช่น อัตราการเรียนรู้เริ่มต้น `lr0` อาจต่ำเกินไป)

### 3. จุดคงตัว (Plateau - การลู่เข้าที่เหมาะสม / สถานการณ์ปกติ A)
- **สิ่งที่คุณเห็น:** การสูญเสียของการฝึกฝนและการตรวจสอบลดลงควบคู่กันและแบนราบในระดับต่ำ ค่า validation mAP ไต่ระดับสูงขึ้นอย่างคงเส้นคงวาและคงที่ในระดับสูง
- **ทำไมจึงเกิดขึ้น:** นี่แสดงถึงพฤติกรรมการฝึกฝนที่เหมาะสมที่สุด แบบจำลองได้เรียนรู้แนวคิดทั่วไปที่ปรับใช้ได้ดีในชุดข้อมูลและได้ลู่เข้าสู่จุดต่ำสุดที่เสถียร
- **สิ่งที่ต้องทำของศาสตราจารย์:** หยุดการฝึกฝน นำส่งออกจุดเช็คพอยต์ `weights/best.pt` ไปยังรูปแบบ ONNX หรือ TensorRT เพื่อใช้ในการใช้งานจริง (deployment)